# Unit 5 — Comparing more than two groups

Unit 4 asked whether **two** groups differ. Here there are three or more, and
the question is the same — but asking it once for every pair of groups stops
working, and the first thing this unit shows is why.

| Outcome | Groups | Omnibus test | Follow-up | Effect size |
|---|---|---|---|---|
| Continuous, roughly normal within groups | Three or more, independent | Welch's one-way ANOVA | Games–Howell | ω² (with η² beside it) |
| Ordinal, or clearly not normal | Three or more, independent | Kruskal–Wallis | Dunn's test, with a stated correction | η²(*H*) |

The six steps from Unit 4 carry over unchanged, with one addition. Step 4 now
has two halves: an **omnibus** test asks whether *any* group differs, and a
**post-hoc** comparison asks *which*.

1. **State** the hypothesis pair.
2. **Inspect** the variables.
3. **Check** the assumptions and decide which test to use.
4. **Run** the omnibus test — then, if it finds something, the post-hoc comparisons.
5. **Report** the result with an effect size.
6. **Conclude** about the hypothesis.

In [ ]:
import math9102 as m9
import numpy as np
import pandas as pd
import pingouin as pg
from scipy import stats

m9.use_house_style()

---

## Why not just run lots of *t*-tests?

Three groups make three pairs. Six groups make fifteen. Run a *t*-test on each
pair at the five per cent level and every one of them has its own one-in-twenty
chance of a false positive. The chance that *at least one* of them is a false
positive — the **familywise error rate** — grows with every pair you add:

In [ ]:
for k in range(3, 8):
    pairs = k * (k - 1) // 2
    print(f"{k} groups, {pairs:2d} pairs: P(at least one false positive) ~ {1 - 0.95 ** pairs:.2f}")

The figure is approximate, because tests that share a group are not
independent, but the direction is not in doubt. Somewhere around six groups, a
batch of *t*-tests is more likely than not to report a difference that is not
there.

ANOVA avoids this by asking **one** question of all the groups at once. That
is what "omnibus" means.

---

## Part 1 — Optimism across three age groups

The wellbeing survey from Unit 4: four hundred and thirty-nine respondents from
Melbourne, with validated scales for optimism, stress, self-esteem and more.

In [ ]:
survey = m9.load_survey()
survey.shape

### Step 1: State the hypothesis

> **H₀:** Mean optimism is the same in all three age groups.
>
> **H₁:** Mean optimism differs in at least one age group.

Read H₁ carefully. It does not say every group differs from every other. One
group differing from the rest is enough to make H₀ false, and that is all a
significant omnibus test can tell you.

The outcome is `toptim`, the total score on the Life Orientation Test. The
grouping variable is `agegp3`, with three bands.

### Step 2: Inspect the variables

In [ ]:
# Step 2. The grouping variable, then the outcome within each group.
# Sorting is not an order: say what the order is once, and reuse it.
ages = ["18 - 29", "30 - 44", "45+"]
m9.frequency(survey, "agegp3")
m9.describe_by(survey, "toptim", "agegp3")

These three bands happen to sort into age order. That is luck, not a rule —
Part 2 has a dataset where alphabetical order puts the youngest group last.
Writing the order out once, as `ages`, means every table, plot and sentence
below uses the same one.

In [ ]:
# What a complete-case analysis costs, on the model variables only.
missing = m9.missingness(survey, ["toptim", "agegp3"])
print(missing.attrs["summary"])
missing

In [ ]:
m9.grouped_box(survey, "toptim", "agegp3", ylabel="Total Optimism", order=ages);

Read the plot before any test. The group means edge upward with age, but the
boxes overlap almost entirely: whatever the test says, expect a small effect.

### Step 3: Check the assumptions

ANOVA assumes the outcome is roughly normal **within each group**, as the
*t*-test did. Check each group, not the pooled outcome.

In [ ]:
for age in ages:
    m9.normality_panel(survey.loc[survey.agegp3 == age, "toptim"],
                       label=f"Total Optimism — age {age}")

In [ ]:
for age in ages:
    display(m9.report_normality(survey[survey.agegp3 == age], "toptim",
                                f"Total Optimism (age {age})"))

Each group is close to the reference line through the middle, and bends at
the top. That bend is a **ceiling**: the scale has a maximum, and a cluster of
respondents sits on it. With well over a hundred people in every group, ANOVA
tolerates a departure of this kind, and the plots show nothing worse. A
parametric test is defensible here — and you should say that it is a
judgement, made from the plots.

#### And equal variances?

The classic ANOVA assumes the groups have equal variances. As with the
*t*-test, **we do not test that assumption and then branch on the result**. We
use Welch's ANOVA, which does not make it, and we *describe* the spread:

In [ ]:
# Step 3. Describe the spread in each group. Do not branch on it.
opt = survey[["toptim", "agegp3"]].dropna()
groups = [opt.loc[opt.agegp3 == a, "toptim"] for a in ages]

sds = [g.std(ddof=1) for g in groups]
print(f"largest SD / smallest SD = {max(sds) / min(sds):.2f}")

levene = stats.levene(*groups, center="median")
print(f"Levene (median-centred): p = {levene.pvalue:.3f}")

You will also meet **Bartlett's test** for the same purpose. It is a poor
choice: it is so sensitive to non-normality that on skewed data it reports
unequal variances that are not there. Here the two happen to agree:

In [ ]:
bartlett = stats.bartlett(*groups)
print(f"Bartlett: p = {bartlett.pvalue:.3f}   Levene: p = {levene.pvalue:.3f}")

The lecture's figure shows what happens when the data are skewed. Whichever
you look at, it is a description of the data, and nothing below depends on it.

### Step 4: Run the omnibus test

In [ ]:
# Step 4. Welch is the default. The classic F is shown only for comparison -
# never choose between them on the outcome of a variance test.
welch = stats.f_oneway(*groups, equal_var=False)
classic = stats.f_oneway(*groups)

print(f"Welch: F = {welch.statistic:.3f}, p = {welch.pvalue:.4f}")
print(f"Classic: F = {classic.statistic:.3f}, p = {classic.pvalue:.4f}")

They agree in verdict, as they usually do. scipy reports *F* and *p* but not
the degrees of freedom; pingouin reports all of them:

In [ ]:
# scipy gives F and p. pingouin also prints Welch's degrees of freedom.
pg.welch_anova(data=opt, dv="toptim", between="agegp3")

**Two degrees of freedom, and the second is not a whole number.** The first is
the number of groups less one. The second is Welch's adjustment for unequal
variances and unequal group sizes, and it is smaller than the classic
*n* − *k*. An *F* is always reported with both: *F*(df₁, df₂).

### Step 4b: Where is the difference?

The omnibus test says the three means are not all equal. It does not say which
differ. Post-hoc comparisons test every pair while holding the familywise error
rate at five per cent.

The follow-up should match the omnibus. **Games–Howell** does not assume equal
variances, so it is the natural partner for Welch's ANOVA:

In [ ]:
# Step 4b. Where is the difference? Games-Howell does not assume equal
# variances, so it is the follow-up that matches Welch's F.
gh = pg.pairwise_gameshowell(data=opt, dv="toptim", between="agegp3")
gh[["A", "B", "diff", "pval", "hedges"]]

**Tukey's HSD** is the equal-variance version, and the one most textbooks show.
Here it agrees:

In [ ]:
# Tukey's HSD assumes equal variances. Shown for comparison only.
print(stats.tukey_hsd(*groups))

Read the table one row at a time. Only one pair has an adjusted *p* below .05:
the youngest group against the oldest. The middle group does not differ
detectably from either neighbour.

A significant omnibus test with a single significant pair is common, and it is
not a contradiction. The omnibus asks whether the three means could plausibly
be equal; the pairs ask a narrower question three times, each with less data.

### Step 5: Effect size

In [ ]:
# Step 5. Eta squared is this sample's share of the variance. Omega squared
# estimates the population's, and is the one to describe in words.
eta2 = m9.eta_squared(groups)
omega2 = m9.omega_squared(groups)

print(f"eta squared = {eta2:.3f}")
print(f"omega squared = {omega2:.3f} ({m9.interpret(omega2, 'eta2')})")

**Eta squared** is the share of the variation in optimism that age group
accounts for *in this sample*. It is always a little too large, because the
group means are fitted to the same data they are judged against. **Omega
squared** corrects for that, so it is the better estimate of the share in the
population. With groups of this size the two are close; with groups of ten,
they are not.

Notice that both come from the classic split of the variation into between and
within — not from Welch's *F*, which has no such split to take a share of. That
is not a mismatch, and every package does the same: the effect size describes
how far apart the groups are, while Welch's adjustment changes only how much the
test trusts that difference when the variances are unequal.

Both read against the same benchmarks: .01 small, .06 medium, .14 large.
Benchmarks, not verdicts.

### Step 6: Report and conclude

In [ ]:
# Step 6. Every number in the sentence comes from the data and the result,
# both degrees of freedom included - Welch's second one is not n - k.
m9.report_anova(survey, "toptim", "agegp3", welch,
                outcome_label="Total Optimism", order=ages)

The post-hoc result needs a sentence too, and nothing stops you generating it
the same way — every number below is read from the table:

In [ ]:
sig = gh[gh.pval < 0.05]
pairs_txt = "; ".join(
    f"{row.B} against {row.A} (difference {row.diff:+.2f}, {m9.fmt_p(row.pval)})"
    for row in sig.itertuples()
)
m9.md(f"Games-Howell comparisons found {len(sig)} of {len(gh)} pairs to differ: {pairs_txt}.")

**Conclusion.** There is evidence to reject H₀: mean optimism is not the same
in all three age groups. The difference that the post-hoc comparisons can
locate lies between the youngest and oldest groups, and the effect is small —
age group accounts for a sliver of the variation in optimism.

Note what this does *not* establish. These are three groups of different people
measured once, not the same people measured as they aged, so the result does
not show that optimism **rises** with age. People born in different decades
may simply differ.

---

## Part 2 — When the omnibus finds nothing

A study of sleep among staff at a Melbourne university. Does sleepiness, on the
Sleepiness and Associated Sensations Scale, differ across three age groups?

In [ ]:
sleep = m9.load_sleep()
m9.describe_by(sleep, "totsas", "agegp3")

Look at the row order. The youngest group, `<= 37`, is printed **last**,
because `<` sorts after the digits. A table in that order invites the reader to
see a trend that is not in the data. Fix it once:

In [ ]:
sleep_ages = ["<= 37", "38 - 50", "51+"]
sl = sleep[["totsas", "agegp3"]].dropna()
sleep_groups = [sl.loc[sl.agegp3 == a, "totsas"] for a in sleep_ages]

> **H₀:** Mean sleepiness (`totsas`) is the same in all three age groups.
>
> **H₁:** Mean sleepiness differs in at least one age group.

In [ ]:
m9.grouped_box(sleep, "totsas", "agegp3", ylabel="Total Sleepiness and Associated Sensations",
               order=sleep_ages);

In [ ]:
sleep_welch = stats.f_oneway(*sleep_groups, equal_var=False)
m9.report_anova(sleep, "totsas", "agegp3", sleep_welch,
                outcome_label="Total Sleepiness and Associated Sensations",
                order=sleep_ages)

**No post-hoc comparisons.** The omnibus test found no evidence that any group
differs, so there is nothing to locate. Running the pairs anyway, hoping one
falls below .05, is exactly the multiple-comparison problem the omnibus test
was there to prevent.

The effect size is still reported. A non-significant result with its effect
size tells the reader how large a difference the data are consistent with;
without it, they cannot tell "no difference" from "too few people to tell".

One more check worth making every time — the result object holds a *p*-value,
and a *p*-value is a probability:

In [ ]:
print(f"F = {sleep_welch.statistic:.3f}, p = {sleep_welch.pvalue:.3f}")
assert 0 <= sleep_welch.pvalue <= 1

That looks too obvious to need saying. Reports with a *p* larger than one, and
an *F* carrying three degrees of freedom, do get written and circulated; the
lecture shows one.
A value pasted into the wrong slot of a sentence template survives because a
typed number looks exactly as authoritative as a correct one.

---

## Part 3 — An ordinal outcome: maths grades and social class

The Youth Cohort Study of England and Wales: thirteen thousand young people who
reached school-leaving age in 2002, with their GCSE grades and their family's
social class.

In [ ]:
youth = m9.load_youthcohort()
m9.frequency(youth, "gradmath")

A GCSE grade is **ordinal**: A* is above A, which is above B, but the gap
between A* and A is not the same thing as the gap between F and G. A mean grade
means nothing. This is a rank test.

### Recoding: write the order out, then check it

The grades are stored as text, and text sorts alphabetically — the table above
puts `A` before `A*`, and `Fail` between `F` and `G`. Map them to codes in the order you mean:

In [ ]:
# An ordinal outcome stored as text. Write the order out in full,
# then check that the recode lost nothing it should not have.
grades = ["A*", "A", "B", "C", "D", "E", "F", "G", "Fail"]
youth["mathcode"] = youth.gradmath.map({g: i + 1 for i, g in enumerate(grades)})

lost = youth.gradmath.notna() & youth.mathcode.isna()
print(f"grades lost to the recode: {lost.sum()}")

**That last line is the point of this section.** Any value the mapping does not
name becomes missing, silently. Here is what happens if the bottom grade is
written the way GCSE results usually print it, as `U` for unclassified, when
this dataset says `Fail`:

In [ ]:
wrong = youth.gradmath.map({g: i + 1 for i, g in enumerate(["A*", "A", "B", "C", "D", "E", "F", "G", "U"])})
dropped = youth.gradmath.notna() & wrong.isna()
print(f"grades lost to the wrong label: {dropped.sum()}")

# Of the students who have a grade, the share in each class that the wrong label loses
youth[youth.gradmath.notna()].assign(dropped=dropped).groupby("famsec").dropped.mean().sort_values()

Every failed grade disappears, and the loss is not even: the lower the social
class, the larger the share of its students who vanish. An analysis run on that
recode would understate the very gradient it was looking for — and nothing
would warn you, because a missing value is not an error. Count before and after
every recode.

### Steps 1–2: Hypothesis and inspection

> **H₀:** The distribution of maths grades is the same in every social class
> group.
>
> **H₁:** In at least one social class group, maths grades tend to be higher or
> lower than in the others.

H₁ is not a restatement of H₀ with the word "no" left in. Check yours: the two
must be contradictory, so that exactly one of them is true.

In [ ]:
classes = [
    "Large employers and higher professionals",
    "Lower professional and higher technical occupations",
    "Intermediate occupations",
    "Lower supervisory occupations",
    "Semi routine and routine occupations",
    "Other",
]
graded = youth.dropna(subset=["mathcode"])
pd.crosstab(graded.famsec, graded.gradmath, normalize="index").reindex(
    index=classes, columns=grades).round(3)

Read along the rows. Add the A* and A columns together and they shrink at every
step down the class scale; add the columns below C and they grow at every step. `Other` is not a rung on
the class scale — it is a separate group, and it is read as one.

### Step 3: Assumptions

Kruskal–Wallis assumes independent observations, an outcome that can at least
be ranked, and — only if you want to talk about medians — groups whose
distributions have the same shape. It does not assume normality.

The dataset also carries a survey weight, `s1weight`, which corrects for
non-response (Unit 3). scipy's rank tests cannot apply it, so this analysis is
**unweighted**: its results describe this sample, and generalising them to the
whole cohort assumes the non-response did not distort the class gradient.
That is a limitation you state, not one you hide.

### Step 4: Run the omnibus test

In [ ]:
# Kruskal-Wallis uses only the ranks, so the codes need only be in order.
# Here 1 is A* and 9 is Fail: a LOW rank is a GOOD grade.
kw = youth.dropna(subset=["mathcode"])
by_class = [kw.loc[kw.famsec == c, "mathcode"] for c in classes]
h = stats.kruskal(*by_class)

eta2_h = m9.eta_squared_h(h.statistic, len(by_class), len(kw))
print(f"H = {h.statistic:.1f}, p = {h.pvalue:.3g}, eta squared (H) = {eta2_h:.3f}")

scipy prints *p* as zero. It is not zero — it is smaller than the computer can
represent — and it is reported as *p* < .001, never as *p* = 0. With thirteen
thousand students it tells you nothing beyond "not chance": the **effect size**
carries the finding. η²(*H*) is the rank-based share of variance, read against the same
benchmarks as η².

### Step 4b: Dunn's test, with a stated correction

Six groups make fifteen pairs. Dunn's test compares every pair on the ranks the
Kruskal–Wallis test already used, and the correction for multiple comparisons
is yours to name:

In [ ]:
# Dunn's test on the same pooled ranks. The correction is not optional:
# fifteen pairs at .05 would turn up a false positive more often than not.
pairs = m9.dunn(kw, "mathcode", "famsec", p_adjust="holm", order=classes)
pairs[["A", "B", "z", "p_unadj", "p_adj"]]

**Bonferroni** multiplies every *p* by the number of pairs. **Holm** does the
same in steps — the smallest *p* by fifteen, the next by fourteen, and so on —
and so it is never less powerful than Bonferroni while protecting against the
same error. Either is defensible; leaving the correction out is not. Here the
two agree, as they will when every *p* is either tiny or large:

In [ ]:
bonferroni = m9.dunn(kw, "mathcode", "famsec", p_adjust="bonferroni", order=classes)
print(f"significant after Holm:       {(pairs.p_adj < 0.05).sum()} of {len(pairs)}")
print(f"significant after Bonferroni: {(bonferroni.p_adj < 0.05).sum()} of {len(pairs)}")

### Steps 5–6: Report and conclude

In [ ]:
m9.report_kruskal(
    kw, "mathcode", "famsec", h,
    outcome_label="GCSE mathematics grade",
    order=classes,
    value_labels={i + 1: g for i, g in enumerate(grades)},
)

The medians are reported as grades, not codes, and where a quartile falls
between two grades the sentence says so rather than rounding.

**Conclusion.** There is evidence to reject H₀: maths grades are not
distributed alike across the social class groups, and the effect is of medium
size. The pattern runs in one direction — students from higher-class families
tend to have higher grades.

Note what this does *not* establish. It is an association in observational data;
it does not show that social class *causes* the difference in grades, and it
cannot separate class from everything that travels with it — school, region,
household income.

---

## Choosing between the tests

In [ ]:
m9.md("""
| Your outcome | Your groups | Omnibus | Follow-up |
|---|---|---|---|
| Continuous, roughly normal within groups | 3 or more | Welch's ANOVA | Games–Howell |
| Ordinal, clearly skewed, or very small groups | 3 or more | Kruskal–Wallis | Dunn, with Holm or Bonferroni |
| Continuous or ordinal | 2 | Unit 4: Welch's *t* or Mann–Whitney | — |
""")

Three habits to carry forward:

- **Run the omnibus first, and follow up only what it found.** Hunting through
  the pairs after a non-significant omnibus is the problem ANOVA exists to
  prevent.
- **Name the correction.** A post-hoc *p*-value means nothing until the reader
  knows how it was adjusted.
- **Write the order of the groups out**, once, and check every recode by
  counting before and after.

## Exercise

`exercise.md` asks you to run both tests on questions you have not seen,
including one where a group is too small to analyse — and to decide what to do
about it.